Find Source and Sink nodes.
Find tranistion nodes where there is only 1 possiblity
See what happens when I restirct to only pia icaos and real icaos that are known to use pia.

In [1]:
from pia.resolver import build_candidates
from datetime import date
from data_engineering.flights.flight_type import FlightIndex, flights_df_to_flights, get_flights, pia_or_american_ladd_icao_filter
import polars as pl
import networkx as nx

from pia.solver import resolve_links_with_margin
start_dt = date(2026, 6, 1)
end_dt = date(2026, 7, 1)
df_flights = get_flights(start_dt, end_dt)
df_flights = df_flights.filter(pia_or_american_ladd_icao_filter())
flights = flights_df_to_flights(df_flights)
flight_index = FlightIndex(flights, None)
candidates = build_candidates(flight_index)
selected_candidates, mapping, df_margins = resolve_links_with_margin(
    flight_index,
    candidates,
)

Loading airports from /Volumes/T2-SSD/planequery/data/raw/ourairports/airports.csv
Found 831635 overlapping ICAO pairs.
Total number of componenets: 7686
Number of selected candidates: 154751
Number of connected components: 10792
Number of connected components: 10792


# Investigating selected_candidates

In [ ]:
from pia.graph import create_graph_from_selected_candidates
from pia.solver import pia_icao_mapping_from_selected_candidates

mapping = pia_icao_mapping_from_selected_candidates(selected_candidates, flight_index)
def selected_graph_stats(graph: nx.DiGraph, flight_index: FlightIndex):
    # I should only get 1 path that contains real_flights[0] and real_flights[-1] maybe 0? 
    pia_icao_mapping_stats = dict()
    selected_flights_by_real_icao = dict()
    for canddiate in selected_candidates:
        previous_flight = flight_index.get_flight_by_id(canddiate.previous_flight_id)
        next_flight = flight_index.get_flight_by_id(canddiate.next_flight_id)
        if not previous_flight.pia:

graph = create_graph_from_selected_candidates(selected_candidates, flight_index)
pia_icao_mapping_stats = selected_graph_stats(graph, flight_index)

In [38]:
flight_index.flights_by_icao["a03c81"][0].flight_id

'a03c81_2026-06-01_16-02'

In [40]:
node = "a03c81_2026-06-01_16-02"
G = graph
roots = [n for n in G if G.in_degree(n) == 0]
leaves = [n for n in G if G.out_degree(n) == 0]

paths = []

for root in roots:
    for left in nx.all_simple_paths(G, root, node):
        for leaf in leaves:
            for right in nx.all_simple_paths(G, node, leaf):
                paths.append(left + right[1:])

In [42]:
paths[0]

['a03c81_2026-06-01_16-02',
 'a03c81_2026-06-01_18-23',
 'a7e945_2026-06-24_15-14',
 'a7e945_2026-06-25_15-56',
 'a7e945_2026-06-30_13-44',
 'a7e945_2026-06-30_17-33',
 'a7e945_2026-06-30_19-03']

In [64]:
pia_icao_mapping_stats = dict()
for nodes in nx.weakly_connected_components(graph):
    component = graph.subgraph(nodes)

    source = next(n for n in component if component.in_degree(n) == 0)
    sink = next(n for n in component if component.out_degree(n) == 0)

    longest_path = nx.dag_longest_path(component)
    path_edges = [
    (u, v, component.edges[u, v])
    for u, v in nx.utils.pairwise(longest_path)
    ]
    score = 0
    for u, v, edge_data in path_edges:
        score += edge_data["score"]
    number_of_flights = len(longest_path)
    # calculate score on this.
    pia_icaos = set()
    for node in longest_path:
        flight = flight_index.get_flight_by_id(node)
        if flight.icao in mapping:
            pia_icaos.add(flight.icao)
    for pia_icao in pia_icaos:
        pia_icao_mapping_stats[pia_icao] = {
            "score": score,
            "number_of_flights": number_of_flights,
        }

In [65]:
len(mapping)

95

In [67]:
len(pia_icao_mapping_stats)

95